# 환경 설정

In [ ]:
from inspect import FrameInfo
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import requests
from io import StringIO
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import xml.etree.ElementTree as ET
import pprint
import warnings

warnings.filterwarnings('ignore')

font_path = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/fonts/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
mpl.rc('font', family = 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

date = datetime.now() - timedelta(days=2)
date_1 = date.strftime('%Y-%m-%d')
date_2 = date.strftime('%Y%m%d')

BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'

Mounted at /content/drive


# 주요 데이터 수집

### 따릉이 대여/반납 이력 데이터(시간대별, 스테이션별)

In [ ]:
api_key = '71474b655273796536324f6f616a61'
bike_rent_rows = []
start = 1
end = 1000

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/json/tbCycleRentData/{start}/{end}/{date_1}/1/'
    try:
        response = requests.get(url, timeout=10).json()
        if 'rentData' in response and 'row' in response['rentData']:
            rows = response['rentData']['row']
            bike_rent_rows.extend(rows)
            start += 1000
            end += 1000
        else:
            break
    except Exception as e:
        break

bike_rent_df = pd.DataFrame(bike_rent_rows)

display(bike_rent_df.head())
bike_rent_df.info()

DATA_PATH = BASE + 'seoul_bike_rent_history.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
bike_rent_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

bike_rent_columns = {
    'BIKE_ID': '자전거ID', 'RENT_DT': '대여일시', 'RENT_ID': '대여소번호',
    'RENT_NM': '대여소명', 'RENT_HOLD': '대여거치대', 'RTN_DT': '반납일시',
    'RTN_ID': '반납대여소번호', 'RTN_NM': '반납대여소명', 'RTN_HOLD': '반납거치대',
    'USE_MIN': '사용시간(분)', 'USE_DST': '이용거리(M)', 'USR_CLS_CD': '사용자종류',
    'SEX_CD': '성별', 'BIRTH_YEAR': '생년', 'RENT_STATION_ID': '대여대여소ID',
    'RETURN_STATION_ID': '반납대여소ID', 'BIKE_SE_CD': '자전거구분'
}

,BIKE_ID,RENT_DT,RENT_ID,RENT_NM,RENT_HOLD,RTN_DT,RTN_ID,RTN_NM,RTN_HOLD,USE_MIN,USE_DST,USR_CLS_CD,SEX_CD,BIRTH_YEAR,RENT_STATION_ID,RETURN_STATION_ID,BIKE_SE_CD,START_INDEX,END_INDEX,RNUM
0,SPB-74728,2026-04-04 01:00:04,01667,중계중학교,0,2026-04-04 01:01:44,01666,노원소방서인근,0,1,199.23,USR_001,M,2009,ST-1284,ST-1283,일반자전거,0,0,1
1,SPB-32557,2026-04-04 01:00:32,01911,구로디지털단지역 앞,0,2026-04-04 01:02:53,02012,대림사거리(동작상떼빌아파트),0,2,0.00,USR_001,M,1994,ST-668,ST-727,일반자전거,0,0,2
2,SPB-83142,2026-04-04 01:02:08,00529,장한평역 8번 출구 앞,99,2026-04-04 01:03:03,04385,용답동 우체국 주변,99,0,420.00,USR_001,F,2003,ST-243,ST-3099,새싹자전거,0,0,3
3,SPB-51668,2026-04-04 01:01:22,01964,원메디타운 앞,0,2026-04-04 01:03:22,03929,고척아이파크아파트 205동 앞,0,2,560.00,USR_001,M,1985,ST-1226,ST-3194,일반자전거,0,0,4
4,SPB-74677,2026-04-04 01:01:06,01750,창포원 남쪽 입구,0,2026-04-04 01:03:22,01696,수락리버시티4단지,0,2,384.17,USR_001,M,1974,ST-2095,ST-2062,일반자전거,0,0,5


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1679 entries, 0 to 1678
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   BIKE_ID            1679 non-null   object
 1   RENT_DT            1679 non-null   object
 2   RENT_ID            1679 non-null   object
 3   RENT_NM            1679 non-null   object
 4   RENT_HOLD          1679 non-null   object
 5   RTN_DT             1679 non-null   object
 6   RTN_ID             1672 non-null   object
 7   RTN_NM             1672 non-null   object
 8   RTN_HOLD           1672 non-null   object
 9   USE_MIN            1679 non-null   object
 10  USE_DST            1679 non-null   object
 11  USR_CLS_CD         1679 non-null   object
 12  SEX_CD             1287 non-null   object
 13  BIRTH_YEAR         1643 non-null   object
 14  RENT_STATION_ID    1679 non-null   object
 15  RETURN_STATION_ID  1672 non-null   object
 16  BIKE_SE_CD         1679 non-null   object


### 스테이션 위치 정보 데이터(위도, 경도)

In [ ]:
api_key = '706b77416573796534366a566d426a'
bike_location_rows = []
start = 1
end = 1000
list_total_count = None

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/xml/bikeStationMaster/{start}/{end}/'
    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'

    rows = root.findall('row')
    if not rows:
        break

    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        bike_location_rows.append(row_dict)

    start += 1000
    end += 1000

bike_location_df = pd.DataFrame(bike_location_rows)

display(bike_location_df.head())
bike_location_df.info()

DATA_PATH = BASE + 'seoul_bike_location_history.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
bike_location_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

bike_location_columns = {
    'RNTLS_ID': '대여소_ID', 'ADDR1': '주소1', 'ADDR2': '주소2',
    'LAT': '위도', 'LOT': '경도'
}

,RNTLS_ID,ADDR1,ADDR2,LAT,LOT
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.510380,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.000000,0.000000
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.529900,126.876541


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3411 entries, 0 to 3410
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   RNTLS_ID  3411 non-null   object
 1   ADDR1     3411 non-null   object
 2   ADDR2     1427 non-null   object
 3   LAT       3411 non-null   object
 4   LOT       3411 non-null   object
dtypes: object(5)
memory usage: 133.4+ KB
CSV 파일 저장 완료


### 실시간 따릉이 대여/거치 정보 데이터

In [ ]:
api_key = '425a554e4f73796536336768454176'
real_time_bike_rent_rows = []
start = 1
end = 1000

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/json/bikeList/{start}/{end}/'
    try:
        response = requests.get(url, timeout=10).json()

        if 'rentBikeStatus' in response and 'row' in response['rentBikeStatus']:
            rows = response['rentBikeStatus']['row']
            real_time_bike_rent_rows.extend(rows)
            start += 1000
            end += 1000
        else:
            break
    except Exception as e:
        break

real_time_bike_rent_df = pd.DataFrame(real_time_bike_rent_rows)

display(real_time_bike_rent_df.head())
print(f"현재 수집된 실시간 데이터 개수: {len(real_time_bike_rent_df)}개") # 수집 확인용 프린트
real_time_bike_rent_df.info()

DATA_PATH = BASE + 'real_time_bike_rent.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
real_time_bike_rent_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

real_time_bike_rent_columns = {
    "stationId": "대여소ID", "stationName": "대여소명", "rackTotCnt": "거치대개수",
    "parkingBikeTotCnt": "자전거주차총건수", "shared": "거치율",
    "stationLatitude": "위도", "stationLongitude": "경도"
}

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,15,102. 망원역 1번출구 앞,10,67,37.55564880,126.91062927,ST-4
1,14,103. 망원역 2번출구 앞,11,79,37.55495071,126.91083527,ST-5
2,13,104. 합정역 1번출구 앞,13,100,37.55073929,126.91508484,ST-6
3,5,105. 합정역 5번출구 앞,15,300,37.55000687,126.91482544,ST-7
4,12,106. 합정역 7번출구 앞,19,158,37.54864502,126.91282654,ST-8


현재 수집된 실시간 데이터 개수: 2710개
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2710 entries, 0 to 2709
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   rackTotCnt         2710 non-null   object
 1   stationName        2710 non-null   object
 2   parkingBikeTotCnt  2710 non-null   object
 3   shared             2710 non-null   object
 4   stationLatitude    2710 non-null   object
 5   stationLongitude   2710 non-null   object
 6   stationId          2710 non-null   object
dtypes: object(7)
memory usage: 148.3+ KB
CSV 파일 저장 완료


### 대여소 위치 및 기상청 격자 좌표 통합 데이터

In [ ]:
target_cols = ['RNTLS_ID', 'LAT', 'LOT', 'ADDR1']
exist_cols = [col for col in target_cols if col in bike_location_df.columns]
station_mapping_df = bike_location_df[exist_cols].copy()

if 'ADDR1' in station_mapping_df.columns:
    station_mapping_df['district'] = station_mapping_df['ADDR1'].apply(
        lambda x: str(x).split()[1] if pd.notnull(x) and len(str(x).split()) > 1 else np.nan
    )
    station_mapping_df['district'] = station_mapping_df['district'].apply(
        lambda x: str(x) if str(x).endswith('구') else np.nan
    )
    station_mapping_df = station_mapping_df.drop(columns=['ADDR1'])

kma_grid_coord_dict = {
    '강남구': (61, 126), '강동구': (62, 126), '강북구': (61, 128),
    '강서구': (58, 126), '관악구': (59, 125), '광진구': (62, 126),
    '구로구': (58, 125), '금천구': (59, 124), '노원구': (61, 129),
    '도봉구': (61, 129), '동대문구': (61, 127), '동작구': (59, 125),
    '마포구': (59, 127), '서대문구': (59, 127), '서초구': (61, 125),
    '성동구': (61, 127), '성북구': (61, 127), '송파구': (62, 126),
    '양천구': (58, 126), '영등포구': (58, 126), '용산구': (60, 126),
    '은평구': (59, 127), '종로구': (60, 127), '중구': (60, 127),
    '중랑구': (62, 128)
}

kma_grid_coord_df = pd.DataFrame.from_dict(kma_grid_coord_dict, orient='index', columns=['grid_x', 'grid_y']).reset_index()
kma_grid_coord_df = kma_grid_coord_df.rename(columns={'index': 'district'})
station_mapping_df = pd.merge(station_mapping_df, kma_grid_coord_df, on='district', how='left')

display(station_mapping_df.head())
station_mapping_df.info()

station_mapping_columns = {
    'RNTLS_ID': '대여소_ID', 'LAT': '위도', 'LOT': '경도',
    'district': '자치구', 'grid_x': '격자X', 'grid_y': '격자Y'
}

,RNTLS_ID,LAT,LOT,district,grid_x,grid_y
0,ST-10,37.552746,126.918617,마포구,59.0,127.0
1,ST-100,37.536667,127.073593,광진구,62.0,126.0
2,ST-1000,37.510380,126.866798,양천구,58.0,126.0
3,ST-1001,0.000000,0.000000,양천구,58.0,126.0
4,ST-1002,37.529900,126.876541,양천구,58.0,126.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3411 entries, 0 to 3410
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   RNTLS_ID  3411 non-null   object 
 1   LAT       3411 non-null   object 
 2   LOT       3411 non-null   object 
 3   district  3407 non-null   object 
 4   grid_x    3407 non-null   float64
 5   grid_y    3407 non-null   float64
dtypes: float64(2), object(4)
memory usage: 160.0+ KB


# 외부 요인 데이터 수집

## 미세먼지 데이터

### 실시간 미세먼지 데이터

In [ ]:
api_key = '706b77416573796534366a566d426a'
start = 1
end = 1000
url = f'http://openAPI.seoul.go.kr:8088/{api_key}/xml/RealtimeCityAir/{start}/{end}/'
real_time_air_rows = []

try:
    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    for row in root.findall('row'):
        msrstn_nm = row.find('MSRSTN_NM').text if row.find('MSRSTN_NM') is not None else None
        pm = row.find('PM').text if row.find('PM') is not None else None
        fpm = row.find('FPM').text if row.find('FPM') is not None else None

        real_time_air_rows.append({
            'MSRSTN_NM': msrstn_nm,
            'PM': pd.to_numeric(pm, errors='coerce'),
            'FPM': pd.to_numeric(fpm, errors='coerce')
        })

    real_time_air_df = pd.DataFrame(real_time_air_rows)

except Exception as e:
    pass

display(real_time_air_df.head())
real_time_air_df.info()

DATA_PATH = BASE + 'real_time_air.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
real_time_air_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

real_time_air_columns = {
    'MSRSTN_NM': '측정소명', 'PM': '미세먼지', 'FPM': '초미세먼지'
}

,MSRSTN_NM,PM,FPM
0,중구,17,10
1,종로구,20,12
2,용산구,36,15
3,은평구,24,9
4,서대문구,22,10


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   MSRSTN_NM  25 non-null     object
 1   PM         25 non-null     int64 
 2   FPM        25 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 732.0+ bytes
CSV 파일 저장 완료


### 일별 미세먼지 데이터

In [ ]:
years = [2021, 2022, 2023, 2024, 2025]
daily_air_rows = []

for year in years:
    file_path = BASE + f'daily_air_{year}.csv'

    if os.path.exists(file_path):
        try:
            temp_df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            temp_df = pd.read_csv(file_path, encoding='cp949')

        rename_dict = {
            '미세먼지농도(㎍/㎥)': '미세먼지(㎍/㎥)',
            '초미세먼지농도(㎍/㎥)': '초미세먼지(㎍/㎥)'
        }
        temp_df = temp_df.rename(columns=rename_dict)

        daily_air_rows.append(temp_df)

if daily_air_rows:
    daily_air_df = pd.concat(daily_air_rows, ignore_index=True)

    display(daily_air_df.head())
    daily_air_df.info()

    SAVE_PATH = BASE + 'daily_air.csv'
    daily_air_df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
    print("CSV 파일 저장 완료")

,측정일시,권역명,측정소명,이산화질소농도(ppm),오존농도(ppm),일산화탄소농도(ppm),아황산가스농도(ppm),미세먼지(㎍/㎥),초미세먼지(㎍/㎥)
0,20210101,동남권,강남구,0.022,0.017,0.5,0.004,22.0,14.0
1,20210101,동남권,강동구,0.029,0.014,0.5,0.003,30.0,20.0
2,20210101,동북권,강북구,0.025,0.013,0.6,0.003,33.0,19.0
3,20210101,서남권,강서구,0.027,0.018,0.3,0.005,25.0,14.0
4,20210101,서남권,관악구,0.033,0.013,0.6,0.004,21.0,12.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45650 entries, 0 to 45649
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   측정일시          45650 non-null  int64  
 1   권역명           45650 non-null  object 
 2   측정소명          45650 non-null  object 
 3   이산화질소농도(ppm)  45404 non-null  float64
 4   오존농도(ppm)     45452 non-null  float64
 5   일산화탄소농도(ppm)  45118 non-null  float64
 6   아황산가스농도(ppm)  45345 non-null  float64
 7   미세먼지(㎍/㎥)     45450 non-null  float64
 8   초미세먼지(㎍/㎥)    45465 non-null  float64
dtypes: float64(6), int64(1), object(2)
memory usage: 3.1+ MB
CSV 파일 저장 완료


## 기온 데이터

### 실시간 기온 데이터

In [ ]:
api_key = '59d0685ecd6d50267a2e1ae5b66f7f72a1beabfb46b0612ed8dca76e974db0a1'
url = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst'
real_time_climatological_rows = []

now = datetime.now()
if now.minute < 40:
    now = now - timedelta(hours=1)
base_date = now.strftime('%Y%m%d')
base_time = now.strftime('%H00')

unique_grids = station_mapping_df[['district', 'grid_x', 'grid_y']].dropna(subset=['grid_x', 'grid_y']).drop_duplicates()

for _, row in unique_grids.iterrows():
    district = row['district']
    nx = str(int(row['grid_x']))
    ny = str(int(row['grid_y']))

    params = {
        'serviceKey': api_key, 'pageNo': '1', 'numOfRows': '1000',
        'dataType': 'JSON', 'base_date': base_date, 'base_time': base_time,
        'nx': nx, 'ny': ny
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if data.get('response', {}).get('header', {}).get('resultCode') == '00':
            items = data['response']['body']['items']['item']
            t1h_val = None
            for item in items:
                if item['category'] == 'T1H':
                    t1h_val = item['obsrValue']
                    break
            real_time_climatological_rows.append({
                'district': district,
                'T1H': pd.to_numeric(t1h_val, errors='coerce')
            })
    except Exception as e:
        pass

real_time_climatological_df = pd.DataFrame(real_time_climatological_rows)

display(real_time_climatological_df.head())
real_time_climatological_df.info()

DATA_PATH = BASE + 'real_time_climatological.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
real_time_climatological_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

real_time_climatological_columns = {
    'district': '자치구', 'T1H': '기온'
}

,district,T1H
0,마포구,8.9
1,광진구,10.4
2,양천구,10.1
3,은평구,8.9
4,성동구,10.4


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   district  25 non-null     object 
 1   T1H       25 non-null     float64
dtypes: float64(1), object(1)
memory usage: 532.0+ bytes
CSV 파일 저장 완료


### 일별 기온 데이터

In [ ]:
years = [2021, 2022, 2023, 2024, 2025]
daily_climatological_rows = []

for year in years:
    file_path = BASE + f'daily_climatological_{year}.csv'

    if os.path.exists(file_path):
        try:
            temp_df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            temp_df = pd.read_csv(file_path, encoding='cp949')

        daily_climatological_rows.append(temp_df)

if daily_climatological_rows:
    daily_climatological_df = pd.concat(daily_climatological_rows, ignore_index=True)

    display(daily_climatological_df.head())
    daily_climatological_df.info()

    SAVE_PATH = BASE + 'daily_climatological.csv'
    daily_climatological_df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
    print("CSV 파일 저장 완료")

,지점,지점명,일시,기온(°C)
0,108,서울,2021-01-01 01:00,-8.7
1,108,서울,2021-01-01 02:00,-9.1
2,108,서울,2021-01-01 03:00,-9.3
3,108,서울,2021-01-01 04:00,-9.3
4,108,서울,2021-01-01 05:00,-9.7


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43704 entries, 0 to 43703
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   지점      43704 non-null  int64  
 1   지점명     43704 non-null  object 
 2   일시      43704 non-null  object 
 3   기온(°C)  43704 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.3+ MB
CSV 파일 저장 완료


## 강수량 데이터

### 실시간 강수량 데이터

In [ ]:
real_time_precipitation_rows = []

for _, row in unique_grids.iterrows():
    district = row['district']
    nx = str(int(row['grid_x']))
    ny = str(int(row['grid_y']))

    params = {
        'serviceKey': api_key, 'pageNo': '1', 'numOfRows': '1000',
        'dataType': 'JSON', 'base_date': base_date, 'base_time': base_time,
        'nx': nx, 'ny': ny
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if data.get('response', {}).get('header', {}).get('resultCode') == '00':
            items = data['response']['body']['items']['item']
            rn1_val = None
            pty_val = None

            for item in items:
                if item['category'] == 'RN1':
                    rn1_val = item['obsrValue']
                elif item['category'] == 'PTY':
                    pty_val = item['obsrValue']

            real_time_precipitation_rows.append({
                'district': district,
                'RN1': pd.to_numeric(rn1_val, errors='coerce'),
                'PTY': pty_val
            })
    except Exception as e:
        pass

real_time_precipitation_df = pd.DataFrame(real_time_precipitation_rows)

display(real_time_precipitation_df.head())
real_time_precipitation_df.info()

DATA_PATH = BASE + 'real_time_precipitation.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
real_time_precipitation_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

real_time_precipitation_columns = {
    'district': '자치구', 'RN1': '1시간강수량', 'PTY': '강수형태'
}

""


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
CSV 파일 저장 완료


### 일별 강수량 데이터

In [ ]:
years = [2021, 2022, 2023, 2024, 2025]
daily_precipitation_rows = []

for year in years:
    file_path = BASE + f'daily_precipitation_{year}.csv'

    if os.path.exists(file_path):
        try:
            temp_df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            temp_df = pd.read_csv(file_path, encoding='cp949')

        daily_precipitation_rows.append(temp_df)

if daily_precipitation_rows:
    daily_precipitation_df = pd.concat(daily_precipitation_rows, ignore_index=True)

    display(daily_precipitation_df.head())
    daily_precipitation_df.info()

    SAVE_PATH = BASE + 'daily_precipitation.csv'
    daily_precipitation_df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
    print("CSV 파일 저장 완료")

,지점,지점명,일시,강수량(mm)
0,108,서울,2021-01-05 00:00,0.0
1,108,서울,2021-01-05 03:00,0.0
2,108,서울,2021-01-05 06:00,0.0
3,108,서울,2021-01-05 09:00,0.0
4,108,서울,2021-01-06 21:00,2.2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5073 entries, 0 to 5072
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   지점       5073 non-null   int64  
 1   지점명      5073 non-null   object 
 2   일시       5073 non-null   object 
 3   강수량(mm)  5073 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 158.7+ KB
CSV 파일 저장 완료


## 강설/적설량 데이터

### 실시간 강설/적설량 데이터

In [ ]:
api_key = '59d0685ecd6d50267a2e1ae5b66f7f72a1beabfb46b0612ed8dca76e974db0a1'
url = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst'
real_time_snow_rows = []

for _, row in unique_grids.iterrows():
    district = row['district']
    nx, ny = str(int(row['grid_x'])), str(int(row['grid_y']))

    params = {
        'serviceKey': api_key, 'pageNo': '1', 'numOfRows': '1000',
        'dataType': 'JSON', 'base_date': base_date, 'base_time': base_time,
        'nx': nx, 'ny': ny
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if data.get('response', {}).get('header', {}).get('resultCode') == '00':
            items = data['response']['body']['items']['item']
            rn1_val = None
            pty_val = None

            for item in items:
                if item['category'] == 'RN1':
                    rn1_val = item['obsrValue']
                elif item['category'] == 'PTY':
                    pty_val = item['obsrValue']

            real_time_snow_rows.append({
                'district': district,
                'PTY': pty_val,
                'RN1': pd.to_numeric(rn1_val, errors='coerce')
            })
    except Exception as e:
        pass

real_time_snow_df = pd.DataFrame(real_time_snow_rows)

display(real_time_snow_df.head())
real_time_snow_df.info()

DATA_PATH = BASE + 'real_time_snow.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
real_time_snow_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

real_time_snow_columns = {
    'district': '자치구', 'PTY': '강수형태', 'RN1': '1시간강수량'
}

,district,PTY,RN1
0,마포구,0,0.0
1,광진구,0,0.0
2,양천구,0,0.0
3,은평구,0,0.0
4,성동구,0,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   district  25 non-null     object 
 1   PTY       25 non-null     object 
 2   RN1       25 non-null     float64
dtypes: float64(1), object(2)
memory usage: 732.0+ bytes
CSV 파일 저장 완료


### 일별 강설/적설량 데이터

In [ ]:
years = [2021, 2022, 2023, 2024, 2025]
daily_snow_rows = []

for year in years:
    file_path = BASE + f'daily_snow_{year}.csv'

    if os.path.exists(file_path):
        try:
            temp_df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            temp_df = pd.read_csv(file_path, encoding='cp949')

        daily_snow_rows.append(temp_df)

if daily_snow_rows:
    daily_snow_df = pd.concat(daily_snow_rows, ignore_index=True)

    display(daily_snow_df.head())
    daily_snow_df.info()

    SAVE_PATH = BASE + 'daily_snow.csv'
    daily_snow_df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
    print("CSV 파일 저장 완료")

,지점,지점명,일시,적설(cm)
0,108,서울,2021-01-04 23:00,0.0
1,108,서울,2021-01-05 00:00,0.0
2,108,서울,2021-01-05 01:00,0.0
3,108,서울,2021-01-05 02:00,0.0
4,108,서울,2021-01-05 03:00,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1916 entries, 0 to 1915
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   지점      1916 non-null   int64  
 1   지점명     1916 non-null   object 
 2   일시      1916 non-null   object 
 3   적설(cm)  1916 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 60.0+ KB
CSV 파일 저장 완료


## 인구 데이터

### 연령대 및 시간대, 요일별 유동 인구 데이터

In [ ]:
api_key = '774c685578737965363648797a6c75'
flow_pop_rows = []
start = 1
end = 1000
list_total_count = None

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/xml/VwsmSignguFlpopW/{start}/{end}/'
    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'

    rows = root.findall('row')
    if not rows:
        break

    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        flow_pop_rows.append(row_dict)

    start += 1000
    end += 1000

flow_pop_df = pd.DataFrame(flow_pop_rows)

display(flow_pop_df.head())
flow_pop_df.info()

DATA_PATH = BASE + 'flow_pop.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
flow_pop_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

flow_pop_columns = {
    "STDR_YYQU_CD": "기준_년분기_코드", "SIGNGU_CD": "자치구_코드", "SIGNGU_CD_NM": "자치구_코드_명",
    "TOT_FLPOP_CO": "총_유동인구_수", "ML_FLPOP_CO": "남성_유동인구_수", "FML_FLPOP_CO": "여성_유동인구_수",
    "AGRDE_10_FLPOP_CO": "연령대_10_유동인구_수", "AGRDE_20_FLPOP_CO": "연령대_20_유동인구_수",
    "AGRDE_30_FLPOP_CO": "연령대_30_유동인구_수", "AGRDE_40_FLPOP_CO": "연령대_40_유동인구_수",
    "AGRDE_50_FLPOP_CO": "연령대_50_유동인구_수", "AGRDE_60_ABOVE_FLPOP_CO": "연령대_60_이상_유동인구_수",
    "TMZON_00_06_FLPOP_CO": "시간대_00_06_유동인구_수", "TMZON_06_11_FLPOP_CO": "시간대_06_11_유동인구_수",
    "TMZON_11_14_FLPOP_CO": "시간대_11_14_유동인구_수", "TMZON_14_17_FLPOP_CO": "시간대_14_17_유동인구_수",
    "TMZON_17_21_FLPOP_CO": "시간대_17_21_유동인구_수", "TMZON_21_24_FLPOP_CO": "시간대_21_24_유동인구_수",
    "MON_FLPOP_CO": "월_유동인구_수", "TUES_FLPOP_CO": "화_유동인구_수", "WED_FLPOP_CO": "수_유동인구_수",
    "THUR_FLPOP_CO": "목_유동인구_수", "FRI_FLPOP_CO": "금_유동인구_수", "SAT_FLPOP_CO": "토_유동인구_수",
    "SUN_FLPOP_CO": "일_유동인구_수"
}

,STDR_YYQU_CD,SIGNGU_CD,SIGNGU_CD_NM,TOT_FLPOP_CO,ML_FLPOP_CO,FML_FLPOP_CO,AGRDE_10_FLPOP_CO,AGRDE_20_FLPOP_CO,AGRDE_30_FLPOP_CO,AGRDE_40_FLPOP_CO,...,TMZON_14_17_FLPOP_CO,TMZON_17_21_FLPOP_CO,TMZON_21_24_FLPOP_CO,MON_FLPOP_CO,TUES_FLPOP_CO,WED_FLPOP_CO,THUR_FLPOP_CO,FRI_FLPOP_CO,SAT_FLPOP_CO,SUN_FLPOP_CO
0,20244,11410,서대문구,95412427,43271220,52141207,12636065,18857944,15827915,15948911,...,12155647,15497659,11725477,13887907,13978113,13989568,14021133,13902988,12873058,12759658
1,20244,11440,마포구,113763836,51418177,62345659,15937115,23167773,21900941,19405274,...,14616217,20128449,14392166,16017046,16087731,16234139,16249829,16384938,16596707,16193445
2,20244,11200,성동구,73277025,34468611,38808415,9643335,13092408,13464613,12073256,...,9337406,12414692,9035976,10470145,10518016,10575543,10588041,10604166,10316339,10204774
3,20242,11305,강북구,93656261,43035821,50620440,13274486,13240661,12017037,14005656,...,10508420,15227351,12463681,13338443,13184251,13334575,13172644,13161696,13584950,13879704
4,20243,11305,강북구,91978076,42439440,49538635,12737858,12884779,11705176,13819201,...,10434557,15015474,12218433,13026351,13011765,13000983,12962985,12963209,13392134,13620648


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   STDR_YYQU_CD             700 non-null    object
 1   SIGNGU_CD                700 non-null    object
 2   SIGNGU_CD_NM             700 non-null    object
 3   TOT_FLPOP_CO             700 non-null    object
 4   ML_FLPOP_CO              700 non-null    object
 5   FML_FLPOP_CO             700 non-null    object
 6   AGRDE_10_FLPOP_CO        700 non-null    object
 7   AGRDE_20_FLPOP_CO        700 non-null    object
 8   AGRDE_30_FLPOP_CO        700 non-null    object
 9   AGRDE_40_FLPOP_CO        700 non-null    object
 10  AGRDE_50_FLPOP_CO        700 non-null    object
 11  AGRDE_60_ABOVE_FLPOP_CO  700 non-null    object
 12  TMZON_00_06_FLPOP_CO     700 non-null    object
 13  TMZON_06_11_FLPOP_CO     700 non-null    object
 14  TMZON_11_14_FLPOP_CO     700 non-null    o

### 자치구 단위 서울 생활인구 데이터

In [ ]:
LIVING_POP_PATH = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/living_pop.csv'

if os.path.exists(LIVING_POP_PATH):
    living_pop_df = pd.read_csv(LIVING_POP_PATH, encoding='utf-8')

    display(living_pop_df.head())
    living_pop_df.info()

living_pop_columns = {
  "STDR_DE_ID": "기준일ID",
  "TMZON_PD_SE": "시간대구분",
  "ADSTRD_CODE_SE": "자치구코드",
  "TOT_LVPOP_CO": "총생활인구수",
  "MALE_F0T9_LVPOP_CO": "남자0세부터9세생활인구수",
  "MALE_F10T14_LVPOP_CO": "남자10세부터14세생활인구수",
  "MALE_F15T19_LVPOP_CO": "남자15세부터19세생활인구수",
  "MALE_F20T24_LVPOP_CO": "남자20세부터24세생활인구수",
  "MALE_F25T29_LVPOP_CO": "남자25세부터29세생활인구수",
  "MALE_F30T34_LVPOP_CO": "남자30세부터34세생활인구수",
  "MALE_F35T39_LVPOP_CO": "남자35세부터39세생활인구수",
  "MALE_F40T44_LVPOP_CO": "남자40세부터44세생활인구수",
  "MALE_F45T49_LVPOP_CO": "남자45세부터49세생활인구수",
  "MALE_F50T54_LVPOP_CO": "남자50세부터54세생활인구수",
  "MALE_F55T59_LVPOP_CO": "남자55세부터59세생활인구수",
  "MALE_F60T64_LVPOP_CO": "남자60세부터64세생활인구수",
  "MALE_F65T69_LVPOP_CO": "남자65세부터69세생활인구수",
  "MALE_F70T74_LVPOP_CO": "남자70세이상생활인구수",
  "FEMALE_F0T9_LVPOP_CO": "여자0세부터9세생활인구수",
  "FEMALE_F10T14_LVPOP_CO": "여자10세부터14세생활인구수",
  "FEMALE_F15T19_LVPOP_CO": "여자15세부터19세생활인구수",
  "FEMALE_F20T24_LVPOP_CO": "여자20세부터24세생활인구수",
  "FEMALE_F25T29_LVPOP_CO": "여자25세부터29세생활인구수",
  "FEMALE_F30T34_LVPOP_CO": "여자30세부터34세생활인구수",
  "FEMALE_F35T39_LVPOP_CO": "여자35세부터39세생활인구수",
  "FEMALE_F40T44_LVPOP_CO": "여자40세부터44세생활인구수",
  "FEMALE_F45T49_LVPOP_CO": "여자45세부터49세생활인구수",
  "FEMALE_F50T54_LVPOP_CO": "여자50세부터54세생활인구수",
  "FEMALE_F55T59_LVPOP_CO": "여자55세부터59세생활인구수",
  "FEMALE_F60T64_LVPOP_CO": "여자60세부터64세생활인구수",
  "FEMALE_F65T69_LVPOP_CO": "여자65세부터69세생활인구수",
  "FEMALE_F70T74_LVPOP_CO": "여자70세이상생활인구수"
}

,stdr_de_id,tmzon_pd_se,adstrd_code_se,tot_lvpop_co,male_f0t9_lvpop_co,male_f10t14_lvpop_co,male_f15t19_lvpop_co,male_f20t24_lvpop_co,male_f25t29_lvpop_co,male_f30t34_lvpop_co,...,female_f30t34_lvpop_co,female_f35t39_lvpop_co,female_f40t44_lvpop_co,female_f45t49_lvpop_co,female_f50t54_lvpop_co,female_f55t59_lvpop_co,female_f60t64_lvpop_co,female_f65t69_lvpop_co,female_f70t74_lvpop_co,ldadng_dt
0,20240101,0,11110,231940.9131,6080.9363,3260.2185,7037.7728,11720.3719,11453.5946,9398.7762,...,8270.8979,8262.8127,7381.2943,8933.9256,8204.0952,8077.3270,6773.4574,5040.6403,13601.2481,20240106080757
1,20240101,0,11140,182354.5506,5902.9352,2159.7405,3988.1295,8122.6534,9192.2022,8468.1934,...,8025.3593,8376.7658,6437.4373,6552.0269,5470.2929,5710.5258,4853.0913,3468.4969,9611.2688,20240106080757
2,20240101,0,11170,255885.5460,7345.2048,3639.3568,5057.8888,9234.4375,12070.7663,12527.3087,...,11832.5148,13239.7348,10843.0989,10513.5132,9354.6379,8665.0049,7487.8939,5751.9700,15265.4809,20240106080757
3,20240101,0,11200,302502.4326,10581.9842,4442.1591,6422.9086,9393.7985,12003.4389,11298.4993,...,12224.1674,15389.4293,12996.5818,12855.6322,11768.9656,11444.6560,9825.7494,7485.4514,18810.1096,20240106080757
4,20240101,0,11215,360329.9580,11123.9665,5904.0852,11819.2905,15020.4273,16807.9300,14211.2764,...,13959.7815,15187.1613,13256.5885,14758.9090,12738.2737,13649.9078,11219.6742,8644.5426,20246.3334,20240106080757


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219600 entries, 0 to 219599
Data columns (total 33 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   stdr_de_id              219600 non-null  int64  
 1   tmzon_pd_se             219600 non-null  int64  
 2   adstrd_code_se          219600 non-null  int64  
 3   tot_lvpop_co            219600 non-null  float64
 4   male_f0t9_lvpop_co      219600 non-null  float64
 5   male_f10t14_lvpop_co    219600 non-null  float64
 6   male_f15t19_lvpop_co    219600 non-null  float64
 7   male_f20t24_lvpop_co    219600 non-null  float64
 8   male_f25t29_lvpop_co    219600 non-null  float64
 9   male_f30t34_lvpop_co    219600 non-null  float64
 10  male_f35t39_lvpop_co    219600 non-null  float64
 11  male_f40t44_lvpop_co    219600 non-null  float64
 12  male_f45t49_lvpop_co    219600 non-null  float64
 13  male_f50t54_lvpop_co    219600 non-null  float64
 14  male_f55t59_lvpop_co

## 서울시 여가 생활 장소(공원, 하천, 강 등) 위치 데이터

### 서울시 공원 위치 데이터

In [ ]:
api_key = '706b77416573796534366a566d426a'
seoul_parks_location_rows = []
start = 1
end = 1000
list_total_count = None

while True:
    url = f'http://openAPI.seoul.go.kr:8088/{api_key}/xml/SearchParkInfoService/{start}/{end}/'
    response = requests.get(url, timeout=15)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'

    rows = root.findall('row')
    if not rows:
        break

    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        seoul_parks_location_rows.append(row_dict)

    start += 1000
    end += 1000

seoul_parks_location_df = pd.DataFrame(seoul_parks_location_rows)

display(seoul_parks_location_df.head())
seoul_parks_location_df.info()

DATA_PATH = BASE + 'seoul_parks_location.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
seoul_parks_location_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

seoul_parks_location_columns = {
    "P_IDX": "공원번호", "P_PARK": "공원명", "P_ADDR": "주소", "P_ZONE": "지역",
    "P_ADMSTR": "관리부서", "G_LONGITUDE": "X좌표", "G_LATITUDE": "Y좌표",
    "LONGITUDE": "경도", "LATITUDE": "위도", "P_LIST_CONTENT": "개요",
    "AREA": "면적", "P_IMG": "이미지", "TEMPLATE_URL": "상세내용URL"
}

,SN,PARK_NM,PARK_OTLN,AREA,OPEN_YMD,MAIN_FCLT,MAIN_PLNT,GD_DOC,VST_ROAD,UTZTN_REF,IMG,RGN,PARK_ADDR,MNG_DEPT,TELNO,XCRD_G,YCRD_G,XCRD,YCRD,URL
0,1,남산공원,남산공원은 도심에 위치하여 서울시민에게 맑은 공기를 제공하는 자연휴식처이며 산책 꽃...,2896887㎡ 임 야 : 2454140㎡ 녹지대 및 기타시설 : 442747㎡,1968.9.10,기반시설 : 광장 45950㎡ 도로 108530㎡ 산책로 6.7㎞ (북측:3.7㎞/...,소나무 단풍 아카시아 상수리나무 등 191종 2881870주,http://parks.seoul.go.kr/upload/seditorMulti/2...,명동역(남산케이블카 와룡묘 서울애니메이션센터 방면) 도보 명동역 3번출구 → 퍼시픽...,남산공원 차량통행 금지안내 2005년 5월 1일부터 남산공원에 일반승용차 택시 통행...,http://parks.seoul.go.kr/file/info/view.do?fId...,중구,서울특별시 중구 삼일대로 231(예장동),중부공원여가센터,02-3783-5900,198364.107,450395.554,126.9903773,37.5501402,http://parks.seoul.go.kr/template/sub/namsan.do
1,2,길동생태공원,길동생태공원은 생물의 서식처를 제공하고 종다양성을 증진시키며 자연생태계의 생물들을 ...,79258.7㎡,1999.5.20,"생태문화센터, 탐방객안내센터, 반딧불이 체험관, 목재데크, 조류관찰대",소나무 보리수 등 64종 31800주 산국 부들 등 138종 192800본,http://parks.seoul.go.kr/template/common/img/p...,지하철: 9호선 중앙보훈병원역 3번출구에서 시내버스 환승 5호선 강동역 4번출구에서...,매주 월요일은 휴관입니다 사전예약 후 입장하실 수 있습니다. 공원을 깨끗하게 이용합...,http://parks.seoul.go.kr/file/info/view.do?fId...,강동구,서울특별시 강동구 천호대로 1291(길동생태공원),동부공원여가센 공원운영과,02-472-2770/ 02-489-2770,213554.12,448852.675,127.1547791,37.5403935,http://parks.seoul.go.kr/template/sub/gildong.do
2,3,서울대공원,서울대공원은 세계 각국의 야생동물들이 살아 숨 쉬는 서울동물원과 다양한 기후대의 식...,9132690m²,1984.5.1,동물원 식물원 테마가든(장미원 어린이동물원 피크닉장) 치유숲 산림욕장 캠핑장 국립현...,"엘레강스야자, 용설란, 파리지옥, 네펜데스, 장미 등",https://grandpark.seoul.go.kr/conts/contsView/...,지하철 4호선 대공원역 하차(2번 출구) 도보 15분 정도 문의처 02)500-73...,일반사항 공원을 깨끗하게 이용합니다. 대중교통을 이용해 주세요. 기념물 시설물 풀과...,http://parks.seoul.go.kr/file/info/view.do?fId...,과천시,경기도 과천시 대공원광장로 102,서울대공원 전략기획실,02-500-7033,200994.267,437163.981,127.0198465,37.4264494,http://grandpark.seoul.go.kr/
3,4,서울숲,서울숲은 한강과 중랑천이 만나고 한강-용산-남찬-청계천-서울숲-한강으로 연결되는 서...,480994㎡,2005.6.18,문화예술시설: 바닥분수 거울연못 군마상 야외무대 가족마당무대 생태시설: 꽃사슴방사장...,수 목 : 소나무 섬잣나무 계수나무 외 95종 415795주 식물원 : 선인장 등 ...,http://parks.seoul.go.kr/template/common/img/p...,지하철: 분당선 서울숲역 4번출구에서 도보 약 2분 2호선 뚝섬역 8번출구에서 도보...,지하철 분당선 서울숲역 4번출구 도보로 약2분 2호선 뚝섬역 8번출구 도보로 약 1...,http://parks.seoul.go.kr/file/info/view.do?fId...,성동구,서울특별시 성동구 뚝섬로 273 (성수동1가),동부여가센터 서울숲관리사무소,02-460-2905,203695.432,449290.726,127.0417982,37.5430717,http://parks.seoul.go.kr/template/sub/seoulfor...
4,5,월드컵공원,월드컵공원은 서울의 서쪽에 위치하여 1978년부터 1993년까지 15년간 서울시민이...,2284085㎡,2002.5.1,평화의 공원 월드컵공원전시관(879㎡) 유니세프광장(2400㎡) 평화광장(5217㎡...,None,http://parks.seoul.go.kr/template/common/img/p...,지하철 6호선 월드컵경기장역 하차 → 1번 출구로 나온 후 직진 → 큰길(도로)이 ...,일반사항 공원을 깨끗하게 이용합니다. 대중교통을 이용해 주세요. 기념물 시설물 풀과...,http://parks.seoul.go.kr/file/info/view.do?fId...,마포구,서울특별시 마포구 하늘공원로 84(월드컵공원),서부공원여가센터,02-300-5501,190658.07,451598.831,126.878907,37.571805,http://parks.seoul.go.kr/template/sub/worldcup...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 20 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   SN         133 non-null    object
 1   PARK_NM    133 non-null    object
 2   PARK_OTLN  133 non-null    object
 3   AREA       130 non-null    object
 4   OPEN_YMD   126 non-null    object
 5   MAIN_FCLT  128 non-null    object
 6   MAIN_PLNT  93 non-null     object
 7   GD_DOC     103 non-null    object
 8   VST_ROAD   120 non-null    object
 9   UTZTN_REF  122 non-null    object
 10  IMG        130 non-null    object
 11  RGN        132 non-null    object
 12  PARK_ADDR  133 non-null    object
 13  MNG_DEPT   133 non-null    object
 14  TELNO      133 non-null    object
 15  XCRD_G     122 non-null    object
 16  YCRD_G     122 non-null    object
 17  XCRD       132 non-null    object
 18  YCRD       132 non-null    object
 19  URL        25 non-null     object
dtypes: object(20)
memory usage: 20.9

### 서울시 하천 위치 데이터

In [ ]:
api_key = '706b77416573796534366a566d426a'
seoul_river_location_rows = []
start = 1
end = 1000
list_total_count = None

while True:
    url = f'http://openAPI.seoul.go.kr:8088/{api_key}/xml/Rivgcdn/{start}/{end}/'
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        root = ET.fromstring(response.text)
    except Exception as e:
        break

    if root.tag == 'RESULT' and root.find('CODE') is not None:
        break

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'

    rows = root.findall('row')
    if not rows:
        break

    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        seoul_river_location_rows.append(row_dict)

    if len(rows) < 1000:
        break

    start += 1000
    end += 1000

seoul_river_location_df = pd.DataFrame(seoul_river_location_rows)

display(seoul_river_location_df.head())
seoul_river_location_df.info()

DATA_PATH = BASE + 'seoul_river_location.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
seoul_river_location_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

seoul_river_location_columns = {
    "RIVER_CD": "하천코드", "RIVER_NM": "하천명", "RIVER_DIV": "하천구분",
    "RIVER_LVL": "하천등급", "RIVER_LEN": "하천연장(km)", "RIVER_AREA": "유역면적",
    "RIVER_START": "기점", "RIVER_END": "종점"
}

,RVR_BSC_PLAN_CD,RVR_BSC_PLAN_BIZ_NM,FNDG_YR,WATSYS_NM,RVR_NM,MNST_NM,TRBT_NM_CYCL1,TRBT_NM_CYCL2,TRBT_NM_CYCL3,TRBT_NM_CYCL4,...,TS_NM,TS_PLAN_FLOOD_QTY,TS_PLAN_FLOOD,TS_PLAN_RW,LSP_NM,LSP_PLAN_FLOOD_QTY,LSP_PLAN_FLOOD_WATL,LSP_PLAN_RW,RVR_PRLG,RMRK
0,1500321201211,아라천 하천기본계획,None,아라천,연결수로,아라천,None,None,None,None,...,인천광역시 계양구 귤현동 연결수로 기점(No.12),"1,260",6.52,110,인천광역시 계양구 귤현동 연결수로 종점(No.0),"1,275",6.68,110,1.200,아라천_연결수로
1,1025060201510,탄천 등 10개 하천기본계획,None,한강,양재천,한강,탄천,양재천,None,None,...,서울시 서초구 우면동 749,489,27.28,47,"서울시 강남구 대치동 74-4, 탄천(지방) 합류점",646,18.01,193,8.230,양재천
2,1025080201510,탄천 등 10개 하천기본계획,None,한강,여의천,한강,탄천,양재천,여의천,None,...,서울시 서초구 신원동 산35 번지선,67,62.64,8,"서울시 서초구 양재동 237, 양재천(지방) 합류점",231,18.07,87,4.830,여의천
3,1024880201510,탄천 등 10개 하천기본계획,None,한강,고덕천,한강,고덕천,None,None,None,...,"서울시 강동구 상일동 425-7번지, 서울시, 경기도계",127,22.91,48,"서울시 강동구 고덕동 364-3, 한강(국가) 합류점",384,19.84,60,3.540,고덕천
4,1024910201510,탄천 등 10개 하천기본계획,None,한강,성내천,한강,성내천,None,None,None,...,"서울시 송파구 마천동 277-48번지 (서울,경기도계)",36,58.49,5,서울시 송파구 신천동 6번지 한강(국가) 합류점,542,18.56,139,7.710,성내천


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   RVR_BSC_PLAN_CD         45 non-null     object
 1   RVR_BSC_PLAN_BIZ_NM     45 non-null     object
 2   FNDG_YR                 0 non-null      object
 3   WATSYS_NM               45 non-null     object
 4   RVR_NM                  45 non-null     object
 5   MNST_NM                 45 non-null     object
 6   TRBT_NM_CYCL1           42 non-null     object
 7   TRBT_NM_CYCL2           33 non-null     object
 8   TRBT_NM_CYCL3           15 non-null     object
 9   TRBT_NM_CYCL4           2 non-null      object
 10  TRBT_NM_CYCL5           0 non-null      object
 11  TRBT_NM_CYCL6           0 non-null      object
 12  DSGN_YMD                43 non-null     object
 13  RVR_DSGN_BSS_ANCMNT_SN  43 non-null     object
 14  RVR_MNG_ADST            44 non-null     object
 15  TS_NM   

## 서울시 학교 및 직장 위치 데이터

### 서울시 학교(초, 중, 고) 위치 데이터

In [ ]:
api_key = '6f69714e427379653435444263424e'
seoul_school_info_school_rows = []
start = 1
end = 1000
list_total_count = None

while True:
    url = f'http://openAPI.seoul.go.kr:8088/{api_key}/xml/neisSchoolInfoIs/{start}/{end}/'
    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    code_element = root.find('.//CODE')
    if code_element is not None and code_element.text != 'INFO-000':
        break

    if list_total_count is None:
        count_element = root.find('.//list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'

    rows = root.findall('.//row')
    if not rows:
        break

    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        seoul_school_info_school_rows.append(row_dict)

    start += 1000
    end += 1000

seoul_school_info_df = pd.DataFrame(seoul_school_info_school_rows)

display(seoul_school_info_df.head())
seoul_school_info_df.info()

DATA_PATH = BASE + 'seoul_school_info.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
seoul_school_info_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

seoul_school_info_columns = {
    "SCHUL_NM": "학교명", "SCHUL_KND_SC_NM": "학교종류", "ADRES": "주소",
    "HMPG_ADRES": "홈페이지", "ORG_TELNO": "전화번호",
    "LGDNG_WGS84_LAT": "위도", "LGDNG_WGS84_LOT": "경도"
}

,CTPV_EDUO_CD,CTPV_EDUO_NM,STD_SCHL_CD,SCHL_NM,ENG_SCHL_NM,SCHL_KND_NM,LCTN_NM,CMPTNC_OGNZ_NM,FNDN_SE,ROAD_NM_ZIP,...,FXNO,HGSCHL_SE_NM,INDST_SPC_CLAS_EXST_YN,HGSCHL_GNRL_UEM_SE_NM,SPCL_HGSCHL_TYPE_NM,ETEX_SE_NM,DAYT_NGHT_SE_NM,FNDN_YMD,OPNS_ANVS,LOAD_DT
0,B10,서울특별시교육청,7134155,선화예술중학교,Sunhwa Arts Middle School,각종학교(중),서울특별시,서울특별시성동광진교육지원청,사립,04991,...,02-453-4641,None,N,일반계,None,전기,주간,19731201,19730705,20230627
1,B10,서울특별시교육청,7134155,선화예술중학교,Sunhwa Arts Middle School,각종학교(중),서울특별시,서울특별시성동광진교육지원청,사립,04991,...,02-453-4641,None,N,일반계,None,전기,주간,19731201,19730705,20230627
2,B10,서울특별시교육청,7081544,여명학교(중),Yeomyung School,각종학교(중),서울특별시,서울특별시강서양천교육지원청,사립,07531,...,02-888-1676,None,N,None,None,전기,주간,20220301,20220301,20231001
3,B10,서울특별시교육청,7081544,여명학교(중),Yeomyung School,각종학교(중),서울특별시,서울특별시강서양천교육지원청,사립,07531,...,02-888-1676,None,N,None,None,전기,주간,20220301,20220301,20231001
4,B10,서울특별시교육청,7061109,예원학교,Yewon School,각종학교(중),서울특별시,서울특별시중부교육지원청,사립,04518,...,02-727-9099,None,N,일반계,None,전기,주간,19661117,19670527,20230627


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CTPV_EDUO_CD            42 non-null     object
 1   CTPV_EDUO_NM            42 non-null     object
 2   STD_SCHL_CD             42 non-null     object
 3   SCHL_NM                 42 non-null     object
 4   ENG_SCHL_NM             42 non-null     object
 5   SCHL_KND_NM             42 non-null     object
 6   LCTN_NM                 42 non-null     object
 7   CMPTNC_OGNZ_NM          42 non-null     object
 8   FNDN_SE                 42 non-null     object
 9   ROAD_NM_ZIP             42 non-null     object
 10  ROAD_NM_ADDR            42 non-null     object
 11  DADDR                   42 non-null     object
 12  TELNO                   42 non-null     object
 13  HMPG_ADDR               42 non-null     object
 14  CEDU_SE_NM              42 non-null     object
 15  FXNO    

### 서울시 대학교 위치 데이터

In [ ]:
api_key = '706b77416573796534366a566d426a'
seoul_university_info_rows = []
start = 1
end = 1000
list_total_count = None

while True:
    url = f'http://openAPI.seoul.go.kr:8088/{api_key}/xml/SebcCollegeInfoKor/{start}/{end}/'
    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    code_element = root.find('.//CODE')
    if code_element is not None and code_element.text != 'INFO-000':
        break

    if list_total_count is None:
        count_element = root.find('.//list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'

    rows = root.findall('.//row')
    if not rows:
        break

    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        seoul_university_info_rows.append(row_dict)

    start += 1000
    end += 1000

seoul_university_info_df = pd.DataFrame(seoul_university_info_rows)

display(seoul_university_info_df.head())
seoul_university_info_df.info()

DATA_PATH = BASE + 'seoul_university_info.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
seoul_university_info_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

seoul_university_info_columns = {
    "JACHIGU": "자치구", "SCHOOL_NM": "학교명", "SCHOOL_GUBUN": "학교구분",
    "ADDRESS": "주소", "TEL": "전화번호", "HOMEPAGE": "홈페이지"
}

,MAIN_KEY,YEAR,CATE1_NAME,NAME_KOR,BRANCH,STATE,TYPE,POSTCODE,ADD_KOR,ADD_KOR_ROAD,H_KOR_CITY,H_KOR_GU,H_KOR_DONG,TEL,FAX,HP
0,BE_LiST23-0034,2013,일반대학,서울시립대학교,본교,기존,공립,None,서울 동대문구 서울시립대로 163 (전농동 90번지),None,서울특별시,동대문구,휘경2동,02-6490-6114,02-2210-5576,http://www.uos.ac.kr
1,BE_LiST23-0035,2013,전문대학(3년제),서울여자간호대학교,본교,기존,사립,None,서울 서대문구 홍제3동 서울여자간호대학,None,서울특별시,서대문구,홍제2동,02-2287-1700,02-395-8018,http://www.snjc.ac.kr
2,BE_LiST23-0036,2013,일반대학,서울여자대학교,본교,기존,사립,None,서울특별시 노원구 화랑로 621 서울여자대학교,None,서울특별시,노원구,공릉2동,02-970-5114,02-978-7931,http://www.swu.ac.kr
3,BE_LiST23-0037,2013,전문대학(2년제),서일대학교,본교,학교명 변경,사립,None,서울 중랑구 서일대학길 22(면목동 49-3) 서일대학교,None,서울특별시,중랑구,면목제3.8동,02-490-7300,02-493-2576,http://www.seoil.ac.kr
4,BE_LiST23-0038,2013,일반대학,성공회대학교,본교,기존,사립,None,서울 구로구 항동 성공회대학교,None,서울특별시,구로구,오류2동,02-2610-4142,02-2610-4248,http://www.skhu.ac.kr


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   MAIN_KEY      64 non-null     object
 1   YEAR          64 non-null     object
 2   CATE1_NAME    64 non-null     object
 3   NAME_KOR      64 non-null     object
 4   BRANCH        64 non-null     object
 5   STATE         64 non-null     object
 6   TYPE          64 non-null     object
 7   POSTCODE      0 non-null      object
 8   ADD_KOR       64 non-null     object
 9   ADD_KOR_ROAD  0 non-null      object
 10  H_KOR_CITY    64 non-null     object
 11  H_KOR_GU      64 non-null     object
 12  H_KOR_DONG    64 non-null     object
 13  TEL           60 non-null     object
 14  FAX           50 non-null     object
 15  HP            63 non-null     object
dtypes: object(16)
memory usage: 8.1+ KB
CSV 파일 저장 완료


### 서울시 직장 위치 데이터

In [ ]:
BUSINESS_STATUS_PATH = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/seoul_business_status.csv'

if os.path.exists(BUSINESS_STATUS_PATH):
    try:
        seoul_business_status_df = pd.read_csv(BUSINESS_STATUS_PATH, encoding='utf-8')
    except UnicodeDecodeError:
        seoul_business_status_df = pd.read_csv(BUSINESS_STATUS_PATH, encoding='cp949')

    display(seoul_business_status_df.head())
    seoul_business_status_df.info()

,순번,등록번호,상호,대표자,전화번호,팩스번호,주소,등록일자,데이터 기준일
0,1,204161,"주식회사 웨스컴(WESSCOM Co., Ltd.)",엄재훈,NaN,NaN,"서울 마포구 서강로 82, 정상제이엘에스빌딩 402호 (창전동)",2025-06-30,2025-06-30
1,2,204158,주식회사 한성디자인기획,김수엽,02-711-3737,02-711-3737,"서울 마포구 서강로14길 3, 1층 (노고산동)",2025-06-24,2025-06-24
2,3,204159,"주식회사 지아이엘시스템(GIL System Co.,Ltd.)",곽철용,NaN,02-6455-5906,"서울 영등포구 당산로41길 11, 에스케이브이1센터 더블유동 1103호 (당산동4가)",2025-06-24,2025-06-24
3,4,204160,중앙케이블네트워크 주식회사,노상우,02-395-2121,02-395-2121,"서울 서대문구 수색로10길 27, 1,2층 (북가좌동)",2025-06-24,2025-06-24
4,5,204157,"어벤투스모빌리티 주식회사(Aventus Mobility Co.,Ltd.)",김장규,02-2142-1000,02-2142-1001,"서울 송파구 중대로 135, 아이티벤처타워 9층 서관901호 (가락동)",2025-06-19,2025-06-19


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2956 entries, 0 to 2955
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   순번       2956 non-null   int64 
 1   등록번호     2956 non-null   int64 
 2   상호       2956 non-null   object
 3   대표자      2956 non-null   object
 4   전화번호     2906 non-null   object
 5   팩스번호     2897 non-null   object
 6   주소       2956 non-null   object
 7   등록일자     2956 non-null   object
 8   데이터 기준일  2942 non-null   object
dtypes: int64(2), object(7)
memory usage: 208.0+ KB


# 데이터 병합(Merge)

## 주요 데이터 병합(Merge)

In [ ]:
PROCESSED_BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/processed/'

### 실시간 대여소 통합 데이터

In [ ]:
real_time_station_df = pd.merge(
    bike_location_df,
    real_time_bike_rent_df,
    left_on='RNTLS_ID',
    right_on='stationId',
    how='left'
)

cols_to_drop = ['stationId', 'stationName', 'stationLatitude', 'stationLongitude']
real_time_station_df = real_time_station_df.drop(columns=[col for col in cols_to_drop if col in real_time_station_df.columns])

display(real_time_station_df.head())
real_time_station_df.info()

DATA_PATH = PROCESSED_BASE + 'real_time_station.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
real_time_station_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

,RNTLS_ID,ADDR1,ADDR2,LAT,LOT,rackTotCnt,parkingBikeTotCnt,shared
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617,12,9,75
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593,NaN,NaN,NaN
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.510380,126.866798,10,4,40
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.000000,0.000000,NaN,NaN,NaN
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.529900,126.876541,10,41,410


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3411 entries, 0 to 3410
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   RNTLS_ID           3411 non-null   object
 1   ADDR1              3411 non-null   object
 2   ADDR2              1427 non-null   object
 3   LAT                3411 non-null   object
 4   LOT                3411 non-null   object
 5   rackTotCnt         2710 non-null   object
 6   parkingBikeTotCnt  2710 non-null   object
 7   shared             2710 non-null   object
dtypes: object(8)
memory usage: 213.3+ KB
CSV 파일 저장 완료


### 대여 이력 통합 데이터

In [ ]:
seoul_bike_history_df = pd.merge(
    bike_rent_df,
    bike_location_df[['RNTLS_ID', 'LAT', 'LOT', 'ADDR1']],
    left_on='RENT_STATION_ID',
    right_on='RNTLS_ID',
    how='left'
)

seoul_bike_history_df = seoul_bike_history_df.rename(columns={
    'LAT': 'RENT_LAT',
    'LOT': 'RENT_LOT',
    'ADDR1': 'RENT_ADDR'
}).drop(columns=['RNTLS_ID'])

display(seoul_bike_history_df.head())
seoul_bike_history_df.info()

DATA_PATH = PROCESSED_BASE + 'seoul_bike_history.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
seoul_bike_history_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

,BIKE_ID,RENT_DT,RENT_ID,RENT_NM,RENT_HOLD,RTN_DT,RTN_ID,RTN_NM,RTN_HOLD,USE_MIN,...,BIRTH_YEAR,RENT_STATION_ID,RETURN_STATION_ID,BIKE_SE_CD,START_INDEX,END_INDEX,RNUM,RENT_LAT,RENT_LOT,RENT_ADDR
0,SPB-74728,2026-04-04 01:00:04,01667,중계중학교,0,2026-04-04 01:01:44,01666,노원소방서인근,0,1,...,2009,ST-1284,ST-1283,일반자전거,0,0,1,37.654499,127.074387,서울특별시 노원구 덕릉로76길 23
1,SPB-32557,2026-04-04 01:00:32,01911,구로디지털단지역 앞,0,2026-04-04 01:02:53,02012,대림사거리(동작상떼빌아파트),0,2,...,1994,ST-668,ST-727,일반자전거,0,0,2,37.484940,126.901321,서울특별시 구로구 시흥대로 577-2
2,SPB-83142,2026-04-04 01:02:08,00529,장한평역 8번 출구 앞,99,2026-04-04 01:03:03,04385,용답동 우체국 주변,99,0,...,2003,ST-243,ST-3099,새싹자전거,0,0,3,37.561371,127.063660,서울특별시 성동구 용답동 224-3
3,SPB-51668,2026-04-04 01:01:22,01964,원메디타운 앞,0,2026-04-04 01:03:22,03929,고척아이파크아파트 205동 앞,0,2,...,1985,ST-1226,ST-3194,일반자전거,0,0,4,37.497890,126.864502,서울특별시 구로구 경인로 403
4,SPB-74677,2026-04-04 01:01:06,01750,창포원 남쪽 입구,0,2026-04-04 01:03:22,01696,수락리버시티4단지,0,2,...,1974,ST-2095,ST-2062,일반자전거,0,0,5,37.688267,127.048897,서울특별시 도봉구 마들로 916 방문자센터


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1679 entries, 0 to 1678
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   BIKE_ID            1679 non-null   object
 1   RENT_DT            1679 non-null   object
 2   RENT_ID            1679 non-null   object
 3   RENT_NM            1679 non-null   object
 4   RENT_HOLD          1679 non-null   object
 5   RTN_DT             1679 non-null   object
 6   RTN_ID             1672 non-null   object
 7   RTN_NM             1672 non-null   object
 8   RTN_HOLD           1672 non-null   object
 9   USE_MIN            1679 non-null   object
 10  USE_DST            1679 non-null   object
 11  USR_CLS_CD         1679 non-null   object
 12  SEX_CD             1287 non-null   object
 13  BIRTH_YEAR         1643 non-null   object
 14  RENT_STATION_ID    1679 non-null   object
 15  RETURN_STATION_ID  1672 non-null   object
 16  BIKE_SE_CD         1679 non-null   object


## 외부 요인 데이터 병합(Merge)

# 데이터 전처리

## 주요 데이터 전처리

### 따릉이 종류별 현황 데이터

In [ ]:
station_bike_type_df = seoul_bike_history_df.copy()

bike_type_mapping = {
    '일반자전거': 'REGULAR',
    '새싹자전거': 'SPROUT',
    'QR': 'QR'
}
station_bike_type_df['BIKE_SE_CD'] = station_bike_type_df['BIKE_SE_CD'].replace(bike_type_mapping)

station_demand = station_bike_type_df.groupby(['RENT_STATION_ID', 'RENT_NM', 'BIKE_SE_CD']).size().reset_index(name='RENT_CNT')

station_bike_type_demand_df = station_demand.pivot_table(
    index=['RENT_STATION_ID', 'RENT_NM'],
    columns='BIKE_SE_CD',
    values='RENT_CNT',
    fill_value=0
).reset_index()

station_bike_type_demand_df.columns.name = None

bike_types = [col for col in station_bike_type_demand_df.columns if col not in ['RENT_STATION_ID', 'RENT_NM']]

station_bike_type_demand_df['TOT_RENT_CNT'] = station_bike_type_demand_df[bike_types].sum(axis=1)

for b_type in bike_types:
    station_bike_type_demand_df[f'{b_type}_RATIO'] = (station_bike_type_demand_df[b_type] / station_bike_type_demand_df['TOT_RENT_CNT'] * 100).round(2)

display(station_bike_type_demand_df.head())
station_bike_type_demand_df.info()

DATA_PATH = PROCESSED_BASE + 'station_bike_type_demand.csv'
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
station_bike_type_demand_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
print("CSV 파일 저장 완료")

station_bike_type_demand_columns = {
    'RENT_STATION_ID': '대여소ID',
    'RENT_NM': '대여소명',
    'TOT_RENT_CNT': '총대여수',
    'REGULAR': '일반자전거_수요',
    'SPROUT': '새싹자전거_수요',
    'REGULAR_RATIO': '일반자전거_비율(%)',
    'SPROUT_RATIO': '새싹자전거_비율(%)'
}

,RENT_STATION_ID,RENT_NM,REGULAR,SPROUT,TOT_RENT_CNT,REGULAR_RATIO,SPROUT_RATIO
0,ST-10,서교동 사거리,3.0,0.0,3.0,100.0,0.0
1,ST-1000,서부식자재마트 건너편,2.0,0.0,2.0,100.0,0.0
2,ST-1002,서울시 도로환경관리센터,1.0,0.0,1.0,100.0,0.0
3,ST-1004,신정이펜하우스314동,1.0,0.0,1.0,100.0,0.0
4,ST-1005,신트리공원 입구,1.0,0.0,1.0,100.0,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 992 entries, 0 to 991
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RENT_STATION_ID  992 non-null    object 
 1   RENT_NM          992 non-null    object 
 2   REGULAR          992 non-null    float64
 3   SPROUT           992 non-null    float64
 4   TOT_RENT_CNT     992 non-null    float64
 5   REGULAR_RATIO    992 non-null    float64
 6   SPROUT_RATIO     992 non-null    float64
dtypes: float64(5), object(2)
memory usage: 54.4+ KB
CSV 파일 저장 완료


## 외부 요인 데이터 전처리